# DCASE 2026 Task 1 — Heterogeneous Audio Classification
## Baseline (HATR) — Colab 실행 노트북

**실행 전 체크리스트**
- 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
- 데이터셋 다운로드 필요: [BSD10k-v1.2](https://zenodo.org/records/17233904) / [BSD35k-CS](https://zenodo.org/records/19187099)

---

## 0. GPU 확인

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

CUDA available: True
GPU: NVIDIA GeForce RTX 3060
VRAM: 8.6 GB


: 

## 1. 패키지 설치

In [3]:
# Colab에는 torch가 이미 설치되어 있으므로 나머지만 설치
!pip install -q numpy pandas scikit-learn pyyaml

## 2. Google Drive 마운트 (데이터셋 저장 위치)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# 작업 디렉토리 설정
import os
PROJECT_DIR = '/content/drive/MyDrive/DCASE2026'  # ← 본인 Drive 경로로 수정
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print('작업 디렉토리:', os.getcwd())

Mounted at /content/drive
작업 디렉토리: /content/drive/MyDrive/DCASE2026


## 3. 베이스라인 코드 세팅

> **방법 A**: GitHub에서 클론 (권장)
> **방법 B**: 업로드한 파일을 직접 복사

In [9]:
# 방법 A: GitHub 클론
!git clone https://github.com/MTG/dcase2026_task1_baseline.git
os.chdir(os.path.join(PROJECT_DIR, 'dcase2026_task1_baseline'))
print('코드 위치:', os.getcwd())
!ls

fatal: destination path 'dcase2026_task1_baseline' already exists and is not an empty directory.
코드 위치: /content/drive/MyDrive/DCASE2026/dcase2026_task1_baseline
BSD10k_metadata.csv  dataset_utils.py  models.py	 summarize_results.py
build_dataset.py     evaluate.py       __pycache__	 train_test.py
config.yaml	     losses.py	       README.md	 utils.py
data		     main.py	       requirements.txt


In [ ]:
# 방법 B: 직접 업로드한 경우 (방법 A 사용 시 이 셀 스킵)
# 아래 파일들을 PROJECT_DIR에 업로드한 뒤 실행
# files = ['train_test.py', 'models.py', 'losses.py', 'dataset_utils.py',
#          'evaluate.py', 'utils.py', 'build_dataset.py', 'config.yaml',
#          'summarize_results.py', 'main.py']
# os.chdir(PROJECT_DIR)

## 4. config.yaml 경로 설정

데이터셋을 Google Drive의 어느 경로에 저장했느냐에 따라 수정하세요.

In [5]:
# config.yaml을 Colab 환경에 맞게 자동으로 덮어쓰기

DATA_ROOT = '/content/drive/MyDrive/DCASE2026/data'  # ← 데이터셋 저장 경로로 수정
ACTIVE_DATASET = 'BSD10k-v1.2'  # 'BSD10k-v1.2' 또는 'BSD35k-CS'

config_content = f"""# INPUT PATHS
active_dataset: {ACTIVE_DATASET}
datasets:
  BSD10k-v1.2:
    metadata_csv: "{DATA_ROOT}/metadata/BSD10k_metadata.csv"
    audio_emb_folder: "{DATA_ROOT}/features/clap_audio_embeddings"
    text_emb_folder: "{DATA_ROOT}/features/clap_text_embeddings"
    class_names: "{DATA_ROOT}/metadata/BST_description.csv"

  BSD35k-CS:
    metadata_csv: "{DATA_ROOT}/metadata/BSD35k-CS_metadata.csv"
    audio_emb_folder: "{DATA_ROOT}/features/clap_audio_embeddings"
    text_emb_folder: "{DATA_ROOT}/features/clap_text_embeddings"
    class_names: "{DATA_ROOT}/metadata/BST_description.csv"

# OUTPUT PATHS
output_path: "data"
processed_dataset_csv: "processed_dataset.csv"
class_dict_json: "class_dict.json"
top_class_dict_json: "top_class_dict.json"
top_class_subclass_dict_json: "top_class_subclass_dict.json"
"""

with open('config.yaml', 'w') as f:
    f.write(config_content)

print('config.yaml 저장 완료')
!cat config.yaml

config.yaml 저장 완료
# INPUT PATHS
active_dataset: BSD10k-v1.2
datasets:
  BSD10k-v1.2:
    metadata_csv: "/content/drive/MyDrive/DCASE2026/data/metadata/BSD10k_metadata.csv"
    audio_emb_folder: "/content/drive/MyDrive/DCASE2026/data/features/clap_audio_embeddings"
    text_emb_folder: "/content/drive/MyDrive/DCASE2026/data/features/clap_text_embeddings"
    class_names: "/content/drive/MyDrive/DCASE2026/data/metadata/BST_description.csv"

  BSD35k-CS:
    metadata_csv: "/content/drive/MyDrive/DCASE2026/data/metadata/BSD35k-CS_metadata.csv"
    audio_emb_folder: "/content/drive/MyDrive/DCASE2026/data/features/clap_audio_embeddings"
    text_emb_folder: "/content/drive/MyDrive/DCASE2026/data/features/clap_text_embeddings"
    class_names: "/content/drive/MyDrive/DCASE2026/data/metadata/BST_description.csv"

# OUTPUT PATHS
output_path: "data"
processed_dataset_csv: "processed_dataset.csv"
class_dict_json: "class_dict.json"
top_class_dict_json: "top_class_dict.json"
top_class_subclass_dict

## 5. 데이터셋 경로 확인

실제 파일이 있는지 확인합니다.

In [7]:
  import os

  checks = [
      f'{DATA_ROOT}/metadata/BSD10k_metadata.csv',
      f'{DATA_ROOT}/features/clap_audio_embeddings',
      f'{DATA_ROOT}/features/clap_text_embeddings',
  ]

  all_ok = True
  for path in checks:
      exists = os.path.exists(path)
      status = '✅' if exists else '❌ 없음!'
      print(f'{status}  {path}')
      if not exists:
          all_ok = False

  if all_ok:
      print('\n모든 경로 확인 완료. 다음 셀을 실행하세요.')
  else:
      print('\n❌ 위 경로를 확인하고 config.yaml을 수정하세요.')

✅  /content/drive/MyDrive/DCASE2026/data/metadata/BSD10k_metadata.csv
✅  /content/drive/MyDrive/DCASE2026/data/features/clap_audio_embeddings
✅  /content/drive/MyDrive/DCASE2026/data/features/clap_text_embeddings

모든 경로 확인 완료. 다음 셀을 실행하세요.


In [6]:
import os

base = '/content/drive/MyDrive/DCASE2026/data/metadata'

print("파일 목록:")
for f in os.listdir(base):
    print(f'[{f}]')

파일 목록:
[BST_diagram.png]
[BST_description.csv]
[BSD10k_metadata.csv]


## 6. train_test.py 버그 수정 패치

원본 코드에 `config.yaml`에 없는 키(`color_dict_path`)를 참조하는 버그가 있어 패치합니다.

In [10]:
# train_test.py에서 존재하지 않는 config 키 참조 제거
with open('train_test.py', 'r') as f:
    code = f.read()

# color_dict_path 참조 제거 (config.yaml에 없는 키)
code = code.replace(
    'color_dict_path = get_subconfig("color_dict_path")\n',
    '# color_dict_path removed (not in config)\n'
)
code = code.replace(
    'top_color_dict_path = get_subconfig("top_color_dict_path")\n',
    '# top_color_dict_path removed (not in config)\n'
)

with open('train_test.py', 'w') as f:
    f.write(code)

print('패치 완료')

# evaluate.py도 동일하게 패치
with open('evaluate.py', 'r') as f:
    code = f.read()

code = code.replace(
    'color_dict_path = get_subconfig("color_dict_path")\n',
    '# color_dict_path removed\n'
)
code = code.replace(
    'top_color_dict_path = get_subconfig("top_color_dict_path")\n',
    '# top_color_dict_path removed\n'
)

with open('evaluate.py', 'w') as f:
    f.write(code)

print('evaluate.py 패치 완료')

패치 완료
evaluate.py 패치 완료


## 7. DataLoader num_workers 패치 (Colab 호환)

Colab에서 `num_workers=4`는 멀티프로세싱 오류를 일으킬 수 있습니다.

In [11]:
for fname in ['train_test.py', 'evaluate.py']:
    with open(fname, 'r') as f:
        code = f.read()
    # num_workers=4 → num_workers=2 (Colab 안정성)
    code = code.replace('num_workers=4', 'num_workers=2')
    with open(fname, 'w') as f:
        f.write(code)

print('num_workers 패치 완료 (4 → 2)')

num_workers 패치 완료 (4 → 2)


## 8. 데이터셋 빌드

임베딩 파일 경로를 스캔하고 `processed_dataset.csv` 생성

In [12]:
!python build_dataset.py

Examining original data from BSD10k-v1.2:
  Total rows: 10956
  Unique classes: 23
After filtering: 10956
Saved class dictionary to data/class_dict.json
Saved top class dictionary to data/top_class_dict.json
Saved top class subclass dictionary to data/top_class_subclass_dict.json
Saved embedding dataframe to data/processed_dataset.csv
Dataset built with 10956 samples.


In [13]:
# 빌드 결과 확인
import pandas as pd
df = pd.read_csv('data/processed_dataset.csv')
print(f'총 샘플 수: {len(df)}')
print(f'클래스 수: {df["class"].nunique()}')
print('\n클래스 분포:')
print(df['class'].value_counts().to_string())

총 샘플 수: 10956
클래스 수: 23

클래스 분포:
class
fx-o     1204
fx-h      977
sp-s      807
ss-u      714
m-si      711
m-sp      685
fx-n      660
is-p      608
is-w      565
is-s      528
ss-n      391
sp-p      361
is-k      359
m-m       320
is-e      308
fx-ex     298
fx-m      281
ss-s      217
fx-el     208
fx-v      204
ss-i      204
sp-c      175
fx-a      171


## 9. 학습 실행

기본 설정: `both` (audio+text) + `audio` only 모드, 5-fold, 100 epoch

In [ ]:
# 전체 파이프라인 실행 (학습 + 평가 + 요약)
# 약 30~60분 소요 (T4 기준, BSD10k, 5-fold x 2 mode)
!python train_test.py

## 10. 결과 요약

In [ ]:
!python summarize_results.py
!cat model_output/summary_metrics.txt

---
## 11. [선택] 빠른 단일 fold 테스트

전체 5-fold 실행 전에 1개 fold만 빠르게 돌려서 동작을 확인할 때 사용

In [ ]:
import sys, os, json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

# 경로가 PROJECT_DIR/dcase2026_task1_baseline 인지 확인
print('현재 경로:', os.getcwd())

from utils import get_subconfig, set_seed, build_class_to_topclass_mapping
from models import BaseClassifier
from losses import CrossEntropyLoss
from dataset_utils import HATRDataset
from evaluate import evaluate_model
from train_test import train_model, init_weights

# ---- 설정 ----
FOLD_ID    = 0         # 테스트할 fold 번호
MODE       = 'both'    # 'both' | 'audio' | 'text'
NUM_EPOCHS = 10        # 빠른 테스트용 (실제는 100)
BATCH_SIZE = 64
HIDDEN     = 128
DROPOUT    = 0.1
K_FOLDS    = 5
# --------------

seed = set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

with open('data/class_dict.json') as f:
    class_dict = json.load(f)
with open('data/top_class_dict.json') as f:
    top_class_dict = json.load(f)

full_df = pd.read_csv('data/processed_dataset.csv')
labels  = full_df['class_idx'].tolist()
skf     = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=seed)

for fold, (trainval_idx, test_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    if fold != FOLD_ID:
        continue

    trainval_labels = [labels[i] for i in trainval_idx]
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx_rel, val_idx_rel = next(sss.split(np.zeros(len(trainval_labels)), trainval_labels))
    train_idx = [trainval_idx[i] for i in train_idx_rel]
    val_idx   = [trainval_idx[i] for i in val_idx_rel]

    train_df = full_df.iloc[train_idx].reset_index(drop=True)
    val_df   = full_df.iloc[val_idx].reset_index(drop=True)
    test_df  = full_df.iloc[test_idx].reset_index(drop=True)
    print(f'Fold {FOLD_ID}: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
    break

train_loader = DataLoader(HATRDataset(train_df, aug=True,  mask_pct=0.7), batch_size=BATCH_SIZE, shuffle=True,  drop_last=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(HATRDataset(val_df,   aug=False),                batch_size=BATCH_SIZE, shuffle=False,                  num_workers=2, pin_memory=True)
test_loader  = DataLoader(HATRDataset(test_df,  aug=False),                batch_size=BATCH_SIZE, shuffle=False,                  num_workers=2, pin_memory=True)

emb_size_audio = 512 if MODE in ['audio', 'both'] else 0
emb_size_text  = 512 if MODE in ['text',  'both'] else 0

model = BaseClassifier(
    hidden_size=HIDDEN,
    num_classes=len(class_dict),
    emb_size_audio=emb_size_audio,
    emb_size_text=emb_size_text,
    dropout=DROPOUT,
    use_batch_norm=True,
    mode=MODE
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model params: {total_params:,}')

init_weights(model)

out_dir = f'model_output/{MODE}/fold_{FOLD_ID}'
os.makedirs(out_dir, exist_ok=True)

best_acc, history, trained_model = train_model(
    model, train_loader, val_loader, device,
    num_epochs=NUM_EPOCHS,
    lr=0.001,
    classification_criterion=CrossEntropyLoss(),
    output_dir=out_dir,
    scheduler_type='step',
    patience=5,
    early_stopping_factor=3
)

print(f'\nBest val accuracy: {best_acc:.2f}%')

metrics = evaluate_model(
    BaseClassifier,
    os.path.join(out_dir, 'best_model.pth'),
    test_loader, device,
    class_to_topclass=build_class_to_topclass_mapping(class_dict, top_class_dict),
    output_dir=out_dir,
    fold_id=FOLD_ID,
    class_dict=class_dict
)

print('\n===== 최종 결과 =====')
for k, v in metrics.items():
    print(f'  {k}: {v:.2f}%')

현재 경로: /content/drive/MyDrive/DCASE2026/dcase2026_task1_baseline
Device: cuda
Fold 0: train=7011, val=1753, test=2192
Model params: 7,319,513


## 12. 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt
import json

# 방금 학습한 fold 0의 history 로드
with open(f'model_output/{MODE}/fold_0/history.json') as f:
    h = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 학습 손실
axes[0].plot(h.get('train_cls_loss', []), label='train CE loss', color='steelblue')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Validation Accuracy
axes[1].plot(h.get('val_accuracy', []), label='val accuracy', color='coral')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('%')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curve.png', dpi=150)
plt.show()

# Attention 가중치 변화 (mode='both'일 때)
if 'attention_audio' in h:
    fig2, ax2 = plt.subplots(figsize=(10, 3))
    audio_attn = [x if isinstance(x, float) else x[0] for x in h['attention_audio']]
    text_attn  = [x if isinstance(x, float) else x[0] for x in h['attention_text']]
    ax2.plot(audio_attn, label='audio weight α₁', color='steelblue')
    ax2.plot(text_attn,  label='text weight α₂',  color='coral')
    ax2.set_title('Attention Weights over Training')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    ax2.grid(alpha=0.3)
    ax2.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig('attention_curve.png', dpi=150)
    plt.show()

## 13. 결과 저장 (Drive 백업)

In [ ]:
import shutil

# model_output 전체를 Drive에 복사
backup_path = os.path.join(PROJECT_DIR, 'model_output_backup')
if os.path.exists(backup_path):
    shutil.rmtree(backup_path)
shutil.copytree('model_output', backup_path)
print(f'백업 완료: {backup_path}')

---
## 알려진 이슈 & 해결법

| 증상 | 원인 | 해결 |
|------|------|------|
| `KeyError: 'color_dict_path'` | config.yaml에 없는 키 참조 | 셀 6 패치 적용 |
| `RuntimeError: DataLoader worker` 오류 | Colab 멀티프로세싱 제한 | 셀 7 패치 (num_workers=2) |
| `CUDA out of memory` | batch_size 너무 큼 | `BATCH_SIZE=32` 로 줄이기 |
| 세션 끊김 후 재시작 | Colab 런타임 초기화 | 셀 2(Drive 마운트)부터 재실행 |
| `Missing audio embedding` 많이 출력 | 임베딩 경로 불일치 | config.yaml 경로 재확인 |

---
*DCASE 2026 Task 1 — Good luck!*